In [ ]:
conda create -n mixlora python=3.8 -y
conda init bash
source ~/.bashrc
conda activate mixlora
sh setup.sh

In [ ]:
sh scripts/finetune_mixlora.sh input 16 8

In [ ]:
which python && which deepspeed
pip install deepspeed
pip install sentencepiece

In [ ]:
source /etc/network_turbo

In [ ]:
pip install torch==2.0.1+cu118 torchvision==0.15.2+cu118 torchaudio==2.0.2+cu118 -f https://download.pytorch.org/whl/cu118/torch_stable.html
pip install flash-attn==2.0.0.post1 --no-build-isolation  #一直编译错误
export MAX_JOBS=4 && pip install flash-attn==2.0.0.post1 --no-build-isolation #终于成功了 但是编译了特别久


In [ ]:

# 当前在MixLoRA目录下，并且有llava文件夹。问题是我们需要将当前目录添加到Python的模块搜索路径中
export PYTHONPATH="/root/autodl-tmp/MixLoRA:$PYTHONPATH" && echo $PYTHONPATH

python -c "import llava; print('llava module found successfully')"

In [ ]:
# 下载模型
pip install modelscope
python -u /root/autodl-tmp/MixLoRA/download.py

In [ ]:
# 解决模型下载网络问题的方案

# 首先检查已下载的文件
ls -la /root/autodl-tmp/model/

# 方案1：创建一个支持重试的下载脚本
cat > /root/autodl-tmp/MixLoRA/download_retry.py << 'EOF'
from modelscope import snapshot_download
import time
import os

def download_with_retry(model_id, cache_dir, max_retries=5):
    for attempt in range(max_retries):
        try:
            print(f"Attempt {attempt + 1}/{max_retries}")
            model_dir = snapshot_download(
                model_id,
                cache_dir=cache_dir,
                resume_download=True  # 启用断点续传
            )
            print(f"Download completed successfully: {model_dir}")
            return model_dir
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                wait_time = (attempt + 1) * 30  # 递增等待时间
                print(f"Waiting {wait_time} seconds before retry...")
                time.sleep(wait_time)
            else:
                print("All retry attempts failed")
                raise e

if __name__ == "__main__":
    download_with_retry(
        'Xorbits/vicuna-7b-v1.3',
        '/root/autodl-tmp/model'
    )
EOF

# 使用支持重试的脚本下载
python /root/autodl-tmp/MixLoRA/download_retry.py

In [ ]:
# 备用方案：使用huggingface hub下载（如果modelscope不稳定）

# 安装huggingface hub
pip install huggingface_hub

# 创建从huggingface下载的脚本
cat > /root/autodl-tmp/MixLoRA/download_hf.py << 'EOF'
from huggingface_hub import snapshot_download
import os

# 设置环境变量以避免下载问题
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = 'False'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = 'True'

try:
    # 尝试从huggingface下载vicuna模型
    model_dir = snapshot_download(
        repo_id="lmsys/vicuna-7b-v1.3", 
        cache_dir="/root/autodl-tmp/model",
        resume_download=True,
        local_files_only=False
    )
    print(f"Download completed: {model_dir}")
except Exception as e:
    print(f"Huggingface download failed: {e}")
    print("Please use the modelscope retry script instead")
EOF

# 如果modelscope继续失败，可以尝试huggingface
# python /root/autodl-tmp/MixLoRA/download_hf.py

In [ ]:
# 网络诊断和状态检查

# 检查网络连接
ping -c 3 www.modelscope.cn
ping -c 3 huggingface.co

# 检查磁盘空间（模型文件很大，确保有足够空间）
df -h /root/autodl-tmp/

# 检查已下载的文件状态
find /root/autodl-tmp/model -name "*.bin" -ls 2>/dev/null || echo "No .bin files found yet"

# 如果部分文件已下载，显示下载进度
if [ -d "/root/autodl-tmp/model" ]; then
    echo "Model directory contents:"
    du -sh /root/autodl-tmp/model/* 2>/dev/null || echo "Model directory is empty or doesn't exist"
fi

# 检查当前网络状态和速度
curl -o /dev/null -s -w "DNS: %{time_namelookup}s, Connect: %{time_connect}s, Total: %{time_total}s, Speed: %{speed_download} bytes/s\n" https://www.modelscope.cn